<a href="https://colab.research.google.com/github/Leontiy2/ITUniverMachineLearningPro/blob/main/model_evaluation_and_safety.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

BLEU - Bilingual Evalution Understudy

In [1]:
!pip install nltk -q

In [2]:
!pip show nltk

Name: nltk
Version: 3.9.1
Summary: Natural Language Toolkit
Home-page: https://www.nltk.org/
Author: NLTK Team
Author-email: nltk.team@gmail.com
License: Apache License, Version 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: click, joblib, regex, tqdm
Required-by: rouge_score, textblob


In [3]:
import nltk
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Учитель і студент швидко завершили проєкт

reference = [["the", "teacher", "and", "the", "student", "completed", "the", "project", "quickly"]]

candidate_1 = ["a", "student", "and", "the", "teacher", "completed", "the", "project", "quickly"]

candidate_2= ["the", "instructor", "and", "the", "pupil", "finished", "the", "program", "quickly"]

smoothie = SmoothingFunction().method1

In [5]:
print("BLUE for candidate_1:", sentence_bleu(reference, candidate_1, smoothing_function=smoothie))
print("BLUE for candidate_2:", sentence_bleu(reference, candidate_2, smoothing_function=smoothie))

BLUE for candidate_1: 0.4032989116748133
BLUE for candidate_2: 0.06376715693797415


##ROUGE

In [6]:
!pip install rouge_score -q

In [7]:
from rouge_score import rouge_scorer

# reference = "Кирило-Мефодіївське товариство - це перша таємна українська політична організація, що діяла в Києві з грудня 1845 по березень 1847 року"
reference = "The Brotherhood of Saints Cyril and Methodius was the first secret Ukrainian political organisation, which operated in Kyiv from December 1845 to March 1847"

# candidate_1 = "Кирило-Мефодіївське товариство - перша українська політична організація, що діяла в Києві до 1847 року"
candidate_1 = "The Cyril and Methodius Brotherhood was the first Ukrainian political organisation, active in Kyiv until 1847"

# candidate_2 = "Кирило-Мефодіївське товариство - українське підпільне політичне об’єднання"
candidate_2 = "The Brotherhood of Saints Cyril and Methodius – a Ukrainian underground political organisation"

# candidate_3 = "Я купив нову корову, зараз ми йдемо на пробіжку"
candidate_3 = "I've bought a new cow; we're off for a run now"

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

for i, candidate in enumerate([candidate_1, candidate_2, candidate_3], 1):
  # print(f"{i}) {candidate}")
  print(f"\n--- Candidat {i} ---")
  scores = scorer.score(reference, candidate)
  # print(scores)

  for key, value in scores.items():
    print(f"{key}: precision={value.precision:.2f}, recall={value.recall:.2f}, fmeasure={value.fmeasure:.2f}")
    # :.2f - форматування рядка (округлення числа) -> str(2.6789:.2f) =


--- Candidat 1 ---
rouge1: precision=0.88, recall=0.58, fmeasure=0.70
rouge2: precision=0.47, recall=0.30, fmeasure=0.37
rougeL: precision=0.81, recall=0.54, fmeasure=0.65

--- Candidat 2 ---
rouge1: precision=0.83, recall=0.42, fmeasure=0.56
rouge2: precision=0.64, recall=0.30, fmeasure=0.41
rougeL: precision=0.83, recall=0.42, fmeasure=0.56

--- Candidat 3 ---
rouge1: precision=0.00, recall=0.00, fmeasure=0.00
rouge2: precision=0.00, recall=0.00, fmeasure=0.00
rougeL: precision=0.00, recall=0.00, fmeasure=0.00


##Perplexity

In [8]:
!pip install transformers torch -q

In [9]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
import torch

model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2TokenizerFast.from_pretrained(model_name)
model.eval()

def calculate_perplaxity(text):
  encodings = tokenizer(text, return_tensors="pt")
  input_ids = encodings.input_ids
  with torch.no_grad():
    outputs = model(input_ids, labels=input_ids)
    loss = outputs.loss
    # print(outputs)
  parplaxity = torch.exp(loss)
  return parplaxity.item()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
calculate_perplaxity("A cow is grazing in a field")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


82.2362289428711

In [11]:
text1 ="A cow is grazing in a field" # Нормальне речення
text2 = "Colorless green ideas sleep furiously." # Граматично вірне дивне речення
text3 = "Table run quickly purple democracy banana." # Набір слів
text4 = "The quick brown fox jumps over the lazy dog." # Речення з продовженням

for i, text in enumerate([text1, text2, text3, text4], start=1):
  ppl = calculate_perplaxity(text)
  print(f"Perplexity: {ppl:.2f} | Text: {text}")

Perplexity: 82.24 | Text: A cow is grazing in a field
Perplexity: 6413.31 | Text: Colorless green ideas sleep furiously.
Perplexity: 36241.88 | Text: Table run quickly purple democracy banana.
Perplexity: 162.47 | Text: The quick brown fox jumps over the lazy dog.


In [12]:
calculate_perplaxity("A cow are graze at the field")

670.3695678710938